# 1. The data: ERCOT wind output and archived day-ahead forecasts

- **Target:** ERCOT's total hourly wind generation (EIA-930, fuel type WND).
- **Providers:** wind forecasts made about 1 and 2 days ahead by public weather models (Open-Meteo Previous Runs API), at 9 points across West Texas, the Panhandle and the South Texas coast.
- **Alignment:** forecasts are instantaneous at the top of each hour, while EIA reports the average over the hour ending at t. So each forecast is the average of the instants at t-1 and t, labeled hour-ending t. Everything is in UTC.

Before modeling: what's actually in the table?

In [1]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, "..")
from src import wind

DATA = "../data/wind"
FIG = "figures"
os.makedirs(FIG, exist_ok=True)
if not os.path.exists(f"{DATA}/wind_hourly.parquet"):
    raise SystemExit("Run the download first, from the repo root: EIA_API_KEY=... python -m src.wind_data")

INK, INK2, SURFACE, GRID = "#0b0b0b", "#52514e", "#fcfcfb", "#e4e3df"
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold", "legend.frameon": False,
})
pd.set_option("display.width", 160)

In [2]:
t = pd.read_parquet(f"{DATA}/wind_hourly.parquet")
w = t["wind_mw"]
print(f"{len(t):,} hours, {t.index.min():%Y-%m-%d} to {t.index.max():%Y-%m-%d}")
print(f"wind output: {w.isna().sum()} missing hours, min {w.min():,.0f} MW, max {w.max():,.0f} MW")

25,056 hours, 2024-01-01 to 2026-11-09
wind output: 24 missing hours, min 169 MW, max 28,819 MW


**Coverage.** Archives start at different times, and some models aren't in the archive at all.

In [3]:
prefixes = sorted({c.rsplit("_", 2)[0] for c in t.columns if c.startswith(("ws_", "ws10_"))})
wind.coverage(t, prefixes)

,provider,lead,points,first hour,% missing after first,mean wind (m/s)
0,ws_ecmwf_ifs025,d1,9,2024-03-06 19:00:00+00:00,0.0,6.94
1,ws_ecmwf_ifs025,d2,9,2024-03-07 19:00:00+00:00,0.0,6.97
2,ws_gem_seamless,d1,9,NaT,NaN,NaN
3,ws_gem_seamless,d2,9,NaT,NaN,NaN
4,ws_gfs_seamless,d1,9,2024-02-16 07:00:00+00:00,0.0,8.06
5,ws_gfs_seamless,d2,9,2024-02-17 07:00:00+00:00,0.0,8.15
6,ws_icon_seamless,d1,9,2024-02-16 13:00:00+00:00,0.0,7.52
7,ws_icon_seamless,d2,9,2024-02-17 13:00:00+00:00,0.0,7.50
8,ws_jma_seamless,d1,9,NaT,NaN,NaN
9,ws_jma_seamless,d2,9,NaT,NaN,NaN


- GFS and ICON start in mid-February 2024, ECMWF in early March. The benchmark starts when all three exist (7 March 2024).
- Météo-France only starts in August 2026, too late to use.
- GEM, JMA and UKMO have no 100 m data in this table. GEM offers 80 m and 120 m (interpolated to 100 m with `python -m src.wind_data --surface`); JMA and UKMO are 10 m only. The archive API was timing out when I pulled GEM again, so the main benchmark uses GFS, ICON and ECMWF.

**Glitches and gaps in the target.**

In [4]:
print("one-hour jumps of more than 30% of capacity:", len(wind.spikes(w)))
missing = w[w.isna()]
runs = missing.index.to_series().diff().ne(pd.Timedelta("1h")).cumsum()
gaps = missing.index.to_series().groupby(runs).agg(["min", "max", "count"])
gaps[gaps["count"] >= 6]

one-hour jumps of more than 30% of capacity: 0


,min,max,count
hour_end_utc,,,
1,2025-12-05 07:00:00+00:00,2025-12-06 06:00:00+00:00,24


**Fleet growth.** ERCOT keeps adding wind farms, so output in 2026 can exceed anything in the training years. The 99th percentile of output per quarter is a rough capacity proxy:

In [5]:
(w.resample("QE").quantile(0.99) / 1000).round(1).rename("p99 output (GW)").to_frame().T

hour_end_utc,2024-03-31 00:00:00+00:00,2024-06-30 00:00:00+00:00,2024-09-30 00:00:00+00:00,2024-12-31 00:00:00+00:00,2025-03-31 00:00:00+00:00,2025-06-30 00:00:00+00:00,2025-09-30 00:00:00+00:00,2025-12-31 00:00:00+00:00,2026-03-31 00:00:00+00:00,2026-06-30 00:00:00+00:00,2026-09-30 00:00:00+00:00,2026-12-31 00:00:00+00:00
p99 output (GW),26.0,25.9,23.1,25.6,27.2,26.6,24.3,26.1,27.0,28.3,25.2,25.3


The proxy grows only a few percent over the period, but tree models can't extrapolate above their training range. Expect a small under-forecast in 2026; the blending notebook tests a correction.

**Is every feed one model?** Comparing each feed's day-1 and day-2 forecasts for the same hour: a single model should agree with itself on average, with a high correlation between leads.

In [6]:
pd.DataFrame({p: wind.lead_check(t, p) for p in ["ws_gfs_seamless", "ws_icon_seamless", "ws_ecmwf_ifs025"]}).T

,mean d1 - d2 (m/s),correlation
ws_gfs_seamless,-0.09,0.949
ws_icon_seamless,0.02,0.961
ws_ecmwf_ifs025,-0.03,0.963


At 100 m, all three behave like single models. When I pulled the 10 m surface winds, though, the "GFS seamless" 10 m feed differed by 1.31 m/s between its day-1 and day-2 forecasts, steadily every quarter, with a lead-to-lead correlation of 0.87 (against about 0.95 for the others). It was blending a different model at short leads near the surface. Ranking providers by lead time without checking this would compare two different models. The 10 m experiment therefore uses `gfs_global`, which is GFS only. (The 10 m columns come from the `--surface` download and aren't in the default table.)